In [1]:
import jax
import jax.numpy as jnp
import numpy as onp
import jax_sn
%load_ext autoreload
%autoreload 2
jax.config.update("jax_enable_x64", True)
import equinox as eqx
import warp as wp 



In [2]:
from jax_sn.tools.setup_test_mgop import setup_slab_mgop, make_xs_test, setup_layered_slab_mgop

In [14]:

def _build_base_mgop():
    """Build the reference mgop with default (Vmap) operators. Expensive — call once."""
    xs = [make_xs_test(m, n_groups=2, l_order=1) for m in ["fuel", "moderator", "reflector"]]
    return setup_layered_slab_mgop(
        dim=3,
        degree=1,
        tn_quadrature_set=3,
        region_sizes=[1.0, 4.0, 2.0],
        region_resolutions=[1330,1,2],
        region_xs=xs,
        region_sources=[1.0, 0.5, 0.2],
        transverse_specs=[(2.0, 2), (1.523, 3)],
    )



In [15]:

onp.random.seed(2042026)

base_mgop = _build_base_mgop()

b_apply = jnp.array(onp.random.random_sample(base_mgop.transport_operator.out_structure().shape))
xs = base_mgop.domain.cross_sections_indexed.map_to_full_array(0)


In [16]:
from jax_sn.operators.transport_operator.linear_tetrahedron_warp_kernel import TransportOperatorSequentialWarpSingleAngle, TransportOperatorSequentialWarpMultipleAngles
top_2 = TransportOperatorSequentialWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator[0].topological_sorted_angle, 0)
top_ma = TransportOperatorSequentialWarpMultipleAngles(TransportOperatorSequentialWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator.topological_sorted_angle, 0))

@eqx.filter_jit
def aaa(top_2 : TransportOperatorSequentialWarpSingleAngle, b):    
    return jnp.sum(top_2.inv(b))



In [ ]:
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory


_filter_jit_f_to_memory(aaa, top_ma, b_apply)

{'peak': 283.39131927490234,
 'argument': 177.58277893066406,
 'output': 3.0517578125e-05,
 'temp': 105.80860900878906}